# SteerScope Metrics Analysis

Run this notebook from the repository root or from its own directory. Configure input and figure paths in the next cell.

Each analysis returns results keyed by model and layer (for example, `2b/l20` and `9b/l20`).


In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd

# Locate the repository without relying on a machine-specific path.
REPO_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "steerscope/sweep/paper/analysis/metrics.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from steerscope.sweep.paper.analysis import metrics_tables
importlib.reload(metrics_tables)
from steerscope.sweep.paper.analysis import metrics
importlib.reload(metrics)

OUTPUT_DIR_VALUES = [
    REPO_ROOT / "outputs/paper/2b/l20",
    REPO_ROOT / "outputs/paper/9b/l20",
]
METHODS = None  # None = all methods; or ["DiffMean", "LsReFT", ...]
PAPER_RELEVANCE_THRESHOLD = 0.2
GENERALIZATION_MIN_ID_EFFECT = 0.1
MIN_REFERENCE_IMPROVEMENT = 0.1
PAPER_EXCLUDED_METHODS = set()
PAPER_FIGURE_BASE = REPO_ROOT / "steerscope/sweep/paper/paper_figures"
DISPLAY_FIGURES_IN_NOTEBOOK = False

# Analysis writes publication PDFs; display controls inline figures only.
metrics.configure(
    OUTPUT_DIR_VALUES=OUTPUT_DIR_VALUES,
    METHODS=METHODS,
    PAPER_RELEVANCE_THRESHOLD=PAPER_RELEVANCE_THRESHOLD,
    GENERALIZATION_MIN_ID_EFFECT=GENERALIZATION_MIN_ID_EFFECT,
    MIN_REFERENCE_IMPROVEMENT=MIN_REFERENCE_IMPROVEMENT,
    PAPER_EXCLUDED_METHODS=PAPER_EXCLUDED_METHODS,
    PAPER_FIGURE_BASE=PAPER_FIGURE_BASE,
    DISPLAY_FIGURES_IN_NOTEBOOK=DISPLAY_FIGURES_IN_NOTEBOOK,
)


## **Global Overall-best factors**

Overall is the harmonic mean of Concept Expression, Instruction Relevance, and Fluency. Each method selects one factor by its mean Overall score across concepts; that fixed factor is used in every subsequent best-factor analysis.


In [ ]:
global_best_factors = metrics.analyze_global_best_factors();

## **Steering outcomes**

### **Efficacy: Concept Expression**

Concept Expression measures how clearly the output expresses the target concept; higher values are better. Factor-sweep plots retain all evaluated factors, while selected-point analyses use the global factor listed above.


In [ ]:
concept_relevance = metrics.analyze_metric(
    "id_lm_judge",
    "relevance_concept_ratings",
    "Concept Expression",
    ylabel="Concept Expression (higher is better)",
    show_tradeoff=False,
);

### **Composite efficacy–side-effect trade-off**

Composite Side Effect is the equal-weight mean of 11 normalized, baseline-relative metric degradations; positive values indicate worse behavior. Main-text plots retain methods whose maximum mean Concept Expression improvement is at least 0.2, while the appendix includes all methods.


In [ ]:
composite_side_effect_curves = metrics.analyze_composite_side_effect_curves();

In [ ]:
composite_side_effect_best_factor = metrics.analyze_composite_side_effect_best_factor();

### **2B / 9B composite comparison**

The two panels share axis limits and one legend, while each panel applies its model-specific method filter and complete factor coverage.


In [ ]:
composite_side_effect_comparison = metrics.analyze_composite_side_effect_comparison(
    composite_side_effect_curves
);


### **Robustness to metric weighting**

This analysis replaces equal weights with 1,000,000 draws from Dirichlet$(1)$, Dirichlet$(5)$, and Dirichlet$(0.2)$. It reports how often no nonzero-factor point has both higher Concept Expression and lower weighted side effect than Prompt Steering, with Wilson intervals for Monte Carlo error.


In [ ]:
metric_weighting_robustness = metrics.analyze_metric_weighting_robustness(
    composite_side_effect_curves
);


### **Best-factor side-effect rank correlation**

Each metric ranks methods after averaging concept-level scores at the method's global factor; higher-is-better and lower-is-better directions are handled explicitly. Spearman correlations use the common set of methods with all 11 metrics available, without interpolation or factor reselection.


In [ ]:
best_factor_rank_correlation = metrics.analyze_best_factor_side_effect_rankings();
best_factor_rank_comparison = metrics.analyze_best_factor_side_effect_rank_comparison(
    best_factor_rank_correlation
);

## **Side effects**

Each point is a fixed `(method, factor)` operating point, reported as a paired change from the same concept's `factor=0` baseline. Positive vertical values indicate degradation, so points toward the lower right have stronger efficacy with fewer side effects.

### **Language quality: Fluency**

Fluency measures the linguistic quality and readability of the generated response; higher raw scores are better.


In [ ]:
fluency = metrics.analyze_metric(
    "id_lm_judge",
    "fluency_ratings",
    "Fluency",
    ylabel="fluency rating (higher is better)",
);

### **Task performance**

- **SuperGLUE** averages eight language-understanding task scores; higher is better.
- **MMLU** measures multiple-choice knowledge accuracy; higher is better.
- **MATH** measures exact-answer mathematical reasoning accuracy; higher is better.
- **Instruction Relevance** measures how well the response addresses the original request; higher is better.
- **IFEval Prompt Strict** requires every instruction in a prompt to be satisfied; higher is better.
- **IFEval Instruction Strict** scores compliance at the individual-instruction level; higher is better.


In [ ]:
superglue = metrics.analyze_metric(
    "superglue",
    "superglue_score",
    "Understanding — SuperGLUE",
    ylabel="SuperGLUE score (higher is better)",
);

In [ ]:
mmlu = metrics.analyze_metric(
    "mmlu",
    "mmlu_accuracy",
    "Knowledge — MMLU",
    ylabel="MMLU accuracy (higher is better)",
);

In [ ]:
math_accuracy = metrics.analyze_metric(
    "math",
    "math_accuracy",
    "Reasoning — MATH",
    ylabel="MATH accuracy (higher is better)",
);

In [ ]:
instruction_relevance = metrics.analyze_metric(
    "id_lm_judge",
    "relevance_instruction_ratings",
    "Original-instruction relevance",
    ylabel="instruction relevance (higher is better)",
);

In [ ]:
ifeval_prompt_strict = metrics.analyze_metric(
    "ifeval",
    "ifeval_prompt_strict_accuracy",
    "Instruction following — IFEval prompt strict",
    ylabel="prompt-level strict accuracy (higher is better)",
);

In [ ]:
ifeval_instruction_strict = metrics.analyze_metric(
    "ifeval",
    "ifeval_instruction_strict_accuracy",
    "Instruction following — IFEval instruction strict",
    ylabel="instruction-level strict accuracy (higher is better)",
);

### **Safety and reliability**

- **Jailbreak attack success** is the harmful-request success rate; lower is better.
- **Jailbreak false refusal** is the refusal rate on benign requests; lower is better.
- **BBQ Accuracy** measures question-answering accuracy; higher is better.
- **BBQ Ambiguous Absolute Bias** measures bias magnitude in ambiguous contexts; lower is better.
- **BBQ Disambiguated Absolute Bias** measures bias magnitude when context resolves the answer; lower is better.
- **TruthfulQA** measures binary truthfulness accuracy; higher is better.


In [ ]:
jbb_attack_success = metrics.analyze_metric(
    "jailbreakbench",
    "attack_success_rate",
    "JailbreakBench — attack success rate",
    filters={"jbb_split": "harmful"},
    ylabel="attack success rate (lower is better)",
    higher_is_better=False,
);

In [ ]:
jbb_false_refusal = metrics.analyze_metric(
    "jailbreakbench",
    "false_refusal_rate",
    "JailbreakBench — false refusal rate",
    filters={"jbb_split": "benign"},
    ylabel="false refusal rate (lower is better)",
    higher_is_better=False,
);

In [ ]:
bbq_accuracy = metrics.analyze_metric(
    "bbq",
    "bbq_accuracy",
    "Bias — BBQ accuracy",
    ylabel="BBQ accuracy (higher is better)",
    paper_main=False,
);

In [ ]:
bbq_ambiguous_bias = metrics.analyze_metric(
    "bbq",
    "bbq_ambiguous_abs_bias",
    "Bias — BBQ ambiguous absolute bias",
    ylabel="absolute ambiguous bias (lower is better)",
    higher_is_better=False,
);


In [ ]:
bbq_disambiguated_bias = metrics.analyze_metric(
    "bbq",
    "bbq_disambiguated_abs_bias",
    "Bias — BBQ disambiguated absolute bias",
    ylabel="absolute disambiguated bias (lower is better)",
    higher_is_better=False,
);


In [ ]:
truthfulqa = metrics.analyze_metric(
    "truthfulqa",
    "truthfulqa_binary_accuracy",
    "Hallucination / truthfulness — TruthfulQA",
    ylabel="TruthfulQA binary accuracy (higher is better)",
);

### **Best-factor metric profiles**

These panels compare MMLU, MATH, IFEval Prompt Strict, Jailbreak Attack Success, SuperGLUE, and TruthfulQA at each method's global factor. Both axes are baseline-relative percentage-point changes: Concept Expression improves to the right and metric degradation increases upward.


In [ ]:
side_effect_capability_panels = metrics.analyze_side_effect_capability_panels(
    main_output_keys=("9b/l20",),
);


## **Method properties**

### **Generalization**

Generalization compares baseline-relative effects on aligned in-distribution (ID) and out-of-distribution (OOD) prompts at each method's global factor. Concept and Overall retention are respectively `mean(OOD ΔC) / mean(ID ΔC)` and `mean(OOD ΔOverall) / mean(ID ΔOverall)`; the table also reports absolute Concept Expression, Instruction Relevance, Fluency, and Overall scores.


In [ ]:
generalization_retention = metrics.analyze_generalization_retention(scope="overall");

### **Data dependence: sample efficiency and sample sensitivity**

Sample Efficiency measures performance as the number of training examples per concept increases. Sample Sensitivity is the mean concept-level sample standard deviation across five 24-example training subsets; all comparisons use the fixed global factor and deterministic inference.


In [ ]:
sample_efficiency = metrics.analyze_sample_efficiency();

In [ ]:
sample_sensitivity = metrics.analyze_sample_sensitivity();

## **Appendix**

### **SuperGLUE task-level scores**

The main analysis reports the equal-weight average of eight SuperGLUE task scores, while this section reports each task's official score. Factor grids include pointwise 95% confidence intervals from 5,000 paired concept-bootstrap replicates; these intervals describe evaluation-sample precision, not training-seed uncertainty.


In [ ]:
superglue_boolq_score = metrics.analyze_metric(
    "superglue",
    "superglue_boolq_score",
    "SuperGLUE BoolQ score",
    ylabel="SuperGLUE BoolQ score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_cb_score = metrics.analyze_metric(
    "superglue",
    "superglue_cb_score",
    "SuperGLUE CB score",
    ylabel="SuperGLUE CB score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_copa_score = metrics.analyze_metric(
    "superglue",
    "superglue_copa_score",
    "SuperGLUE COPA score",
    ylabel="SuperGLUE COPA score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_multirc_score = metrics.analyze_metric(
    "superglue",
    "superglue_multirc_score",
    "SuperGLUE MultiRC score",
    ylabel="SuperGLUE MultiRC score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_record_score = metrics.analyze_metric(
    "superglue",
    "superglue_record_score",
    "SuperGLUE ReCoRD score",
    ylabel="SuperGLUE ReCoRD score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_rte_score = metrics.analyze_metric(
    "superglue",
    "superglue_rte_score",
    "SuperGLUE RTE score",
    ylabel="SuperGLUE RTE score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_wic_score = metrics.analyze_metric(
    "superglue",
    "superglue_wic_score",
    "SuperGLUE WiC score",
    ylabel="SuperGLUE WiC score (higher is better)",
    paper_main=False,
);

In [ ]:
superglue_wsc_score = metrics.analyze_metric(
    "superglue",
    "superglue_wsc_score",
    "SuperGLUE WSC score",
    ylabel="SuperGLUE WSC score (higher is better)",
    paper_main=False,
);

In [ ]:
paper_appendix_pdfs = metrics.render_registered_metric_appendix();